# EXP_040C — Fusion Ablation: Gated Cross-Modal + MSE
**Phase 4 | Fusion Upgrade**
Research question: Does Gated Cross-Modal Fusion improve over GMU?
- Fusion: `gated_cross` | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Set BEST_IMAGE_EXP_ID and BEST_TEXT_EXP_ID in STEP 4.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13379, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 13379 (delta 185), reused 165 (delta 103), pack-reused 13129 (from 1)
Receiving objects: 100% (13379/13379), 873.22 MiB | 18.94 MiB/s, done.
Resolving deltas: 100% (423/423), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1340
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 23 16:10 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths — ✏️ Set Phase 2 & Phase 3 winners here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_040C_bestimage_besttext_gatedcrossmodal_mse'

BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'  # ✏️ Phase 2 winner
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'
BEST_TEXT_EXP_ID  = 'EXP_030B_bestimage_phobert_concat_mse'  # ✏️ Phase 3 winner
BEST_TEXT_MODEL   = 'vinai/phobert-base-v2'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')
print(f'Image: {BEST_IMAGE_MODEL} | Text: {BEST_TEXT_MODEL}')

Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_040C_bestimage_besttext_gatedcrossmodal_mse
Image: swin_base_patch4_window7_224 | Text: vinai/phobert-base-v2


### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

Loaded text from EXP_030B_bestimage_phobert_concat_mse
Loaded image from EXP_020B_swinb_xlmr_concat_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type gated_cross \
  --text_model_name {BEST_TEXT_MODEL} \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_040C_bestimage_besttext_gatedcrossmodal_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_040C_bestimage_besttext_gatedcrossmodal_mse
config.json: 100% 678/678 [00:00<00:00, 2.89MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 100MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 104MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 152MB/s]
Loaded timm processor for swin_base_patch4_window7_224
pytorch_model.bin: 100% 540M/540M [00:03<00:00, 142MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 20351.67it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from 

### STEP 7: Save to Drive + print metrics

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")


=== EXP_040C_bestimage_besttext_gatedcrossmodal_mse Results ===
Loss (val)   : 2.1740

             MAE      RMSE      R2
  food     : 1.0963   1.4810   0.5832
  price    : 1.1713   1.5658   0.4510
  atmos    : 1.1765   1.5171   0.4070
  service  : 1.1772   1.5618   0.5243
  overall  : 0.9198   1.2259   0.6309

  mean_mae   : 1.1082
  aspect_mae : 1.1553
  overall_mae: 0.9198
